# HAT-P-32b emission retrieval with ROBERT and UltraNest

This notebook runs ROBERT's current validation retrieval: H2O correlated-k opacity is binned onto the observed JWST/NIRSpec G395H bins with `exo_k`, evaluated through ROBERT's cloud-free emission RT, and sampled with UltraNest.

This is a real end-to-end retrieval test, but not yet a publication-grade atmospheric model: it retrieves one constant H2O abundance, a temperature-profile offset, and a radius scale.

## Environment

Start Jupyter from the ROBERT repository and install the optional dependencies into this kernel if necessary:

```bash
python -m pip install -e ".[dev,notebooks,opacity,retrieval]"
```

The validated example settings use 40 live points and a 10,000-call ceiling; this converged in the recorded development run. Convergence still needs to be checked for every changed model, prior, or dataset before interpreting a posterior or evidence.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "robert-matplotlib"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(Path(tempfile.gettempdir()) / "robert-numba-cache"))

import matplotlib.pyplot as plt
import numpy as np
try:
    from IPython.display import display
except ImportError:
    display = print

def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "examples").is_dir():
            return candidate
    raise RuntimeError("Start Jupyter from inside the ROBERT repository.")

REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
sys.path.insert(0, str(REPOSITORY_ROOT / "examples"))

from retrieve_hat_p_32b_rt import (  # noqa: E402
    DEFAULT_KTA_DIR,
    DEFAULT_OBSERVATION_NPZ,
    DEFAULT_PT_CSV,
    build_hat_p_32b_forward_model,
)
from robert_exoplanets import (  # noqa: E402
    RetrievalParameter,
    RetrievalParameterSet,
    RetrievalProblem,
    UniformPrior,
    load_emission_observation_npz,
    run_retrieval,
)

print(f"Repository: {REPOSITORY_ROOT}")

## Configure inputs and sampler

Edit these paths if the HAT-P-32b benchmark data are stored elsewhere. The observation NPZ currently contains bin centres but not explicit edges, so ROBERT infers contiguous midpoint edges before handing them to `exo_k`.

In [ ]:
OBSERVATION_NPZ = DEFAULT_OBSERVATION_NPZ
PT_CSV = DEFAULT_PT_CSV
KTA_DIR = DEFAULT_KTA_DIR
OUTPUT_DIR = REPOSITORY_ROOT / "retrieval_runs" / "hat_p_32b_ultranest_notebook"

LOG_H2O_PRIOR = (-6.0, -2.0)
TEMPERATURE_OFFSET_PRIOR_K = (-400.0, 400.0)
RADIUS_SCALE_PRIOR = (0.8, 1.2)

EXOK_NUM = 300
INCLUDE_RAYLEIGH = True
LIVE_POINTS = 40
MAX_NCALLS = 10000
DLOGZ = 1.5
SEED = 20_260_710
RESUME = "overwrite"

required_inputs = [OBSERVATION_NPZ, PT_CSV, KTA_DIR / "H2O_emission_R1000.kta"]
missing = [str(path) for path in required_inputs if not Path(path).exists()]
if missing:
    raise FileNotFoundError("Missing retrieval inputs:\n" + "\n".join(missing))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results will be written under {OUTPUT_DIR}")

## Build the observation, exo_k-prepared opacity, RT model, and retrieval problem

Opacity preparation happens once here, outside the sampler's likelihood loop.

In [ ]:
observation = load_emission_observation_npz(
    OBSERVATION_NPZ,
    instrument="JWST/NIRSpec G395H",
)
forward_model = build_hat_p_32b_forward_model(
    observation,
    pt_csv=PT_CSV,
    kta_dir=KTA_DIR,
    include_rayleigh=INCLUDE_RAYLEIGH,
    exok_num=EXOK_NUM,
)
parameters = RetrievalParameterSet(
    (
        RetrievalParameter("log_h2o", UniformPrior(*LOG_H2O_PRIOR)),
        RetrievalParameter(
            "temperature_offset",
            UniformPrior(*TEMPERATURE_OFFSET_PRIOR_K),
            unit="K",
        ),
        RetrievalParameter("radius_scale", UniformPrior(*RADIUS_SCALE_PRIOR)),
    )
)
problem = RetrievalProblem(
    name="hat-p-32b-rt-ultranest-notebook",
    observation=observation,
    parameters=parameters,
    forward_model=forward_model,
    metadata={**dict(forward_model.manifest_metadata), "interface": "jupyter_notebook"},
    opacity_identifiers=forward_model.opacity_identifiers,
)

print(f"Observed points: {observation.n_points}")
print(f"Spectral bins: {observation.spectral_grid.bin_edges.size - 1}")
print(f"Opacity checksums: {dict(problem.opacity_identifiers)}")

## Forward-model sanity check

Evaluate the prior midpoint before committing to sampling. This catches missing opacity coverage or RT failures cheaply.

In [ ]:
initial_vector = parameters.midpoint_vector()
initial_parameters = parameters.vector_to_mapping(initial_vector)
initial_model = problem.model_spectrum(initial_parameters)
initial_loglike = problem.log_likelihood_from_vector(initial_vector)

fig, ax = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
ax.errorbar(
    observation.wavelength,
    observation.flux * 1e6,
    yerr=observation.uncertainty * 1e6,
    fmt=".",
    color="black",
    ecolor="0.65",
    label="Observed G395H",
)
ax.plot(initial_model.spectral_grid.values, initial_model.values * 1e6, label="Prior midpoint RT")
ax.set(xlabel="Wavelength [micron]", ylabel="Eclipse depth [ppm]", title=f"Initial log L = {initial_loglike:.2f}")
ax.grid(alpha=0.25)
ax.legend(frameon=False);

## Run UltraNest

ROBERT writes `manifest.json` before sampling, then `result.json` and `result_arrays.npz`. Re-running with `RESUME = "overwrite"` replaces UltraNest's sampler directory; choose a new output directory or an appropriate UltraNest resume mode for long runs.

In [ ]:
result = run_retrieval(
    problem,
    method="ultranest",
    output_dir=OUTPUT_DIR,
    seed=SEED,
    min_num_live_points=LIVE_POINTS,
    max_ncalls=MAX_NCALLS,
    dlogz=DLOGZ,
    resume=RESUME,
    mpi_nprocs=1,
    show_status=True,
)
result

## Inspect the persisted result and fit

Do not interpret `log_evidence` when `converged` is false. A bounded smoke test that reaches `MAX_NCALLS` is expected to report `maximum likelihood-call limit reached`.

In [ ]:
summary = json.loads((OUTPUT_DIR / "result.json").read_text(encoding="utf-8"))
manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text(encoding="utf-8"))
best_model = problem.model_spectrum(result.best_fit_parameters)
residual = observation.flux - best_model.values
chi2 = float(np.sum((residual / observation.uncertainty) ** 2))
reduced_chi2 = chi2 / (observation.n_points - problem.ndim)

display({
    "converged": result.converged,
    "message": result.message,
    "best_fit_parameters": dict(result.best_fit_parameters),
    "best_fit_log_likelihood": result.best_fit_log_likelihood,
    "reduced_chi2": reduced_chi2,
    "log_evidence": result.log_evidence,
    "log_evidence_error": result.log_evidence_error,
    "config_hash": manifest["config_hash"],
    "opacity_identifiers": manifest["opacity_identifiers"],
})

In [ ]:
fig, (ax, residual_ax) = plt.subplots(
    2, 1, figsize=(9, 6.5), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
    constrained_layout=True,
)
ax.errorbar(
    observation.wavelength, observation.flux * 1e6,
    yerr=observation.uncertainty * 1e6, fmt=".",
    color="black", ecolor="0.65", label="Observed G395H",
)
ax.plot(best_model.spectral_grid.values, best_model.values * 1e6, color="#f58518", label="ROBERT best fit")
ax.set(ylabel="Eclipse depth [ppm]", title=f"HAT-P-32b UltraNest retrieval — reduced χ² = {reduced_chi2:.3f}")
ax.grid(alpha=0.25)
ax.legend(frameon=False)

residual_ax.axhline(0.0, color="0.4", linewidth=1)
residual_ax.errorbar(
    observation.wavelength, residual / observation.uncertainty,
    yerr=np.ones(observation.n_points), fmt=".", color="#4c78a8", ecolor="0.75",
)
residual_ax.set(xlabel="Wavelength [micron]", ylabel="Residual [σ]")
residual_ax.grid(alpha=0.25);

## Before treating this as a science retrieval

1. Supply measured wavelength-bin edges rather than midpoint-inferred edges.
2. Increase the sampler budget and live points until UltraNest converges, then repeat with multiple seeds.
3. Expand the atmospheric model beyond constant H2O and a uniform temperature offset.
4. Validate the resulting spectrum against an independent RT implementation and injection-recovery fixtures.
5. Add correlated/noise-systematics models appropriate to the reduced JWST product.